# MODEL GOVERNANCE & MONITORING

### Objective

Ensure the deployed credit risk model remains accurate, fair, and financially reliable over time through continuous monitoring and governance controls.

Why Model Governance Is Critical

- Customer behavior changes

- Economic cycles shift default patterns

- Models degrade silently

- Regulatory scrutiny requires traceability

📌 A model without governance is a future liability.

In [7]:
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from scipy.stats import ks_2samp

In [9]:
bundle = joblib.load("gb_bundle (1).pkl")

gb_model = bundle["model"]
train_scores = bundle["train_proba"]
test_scores = bundle["test_proba"]
y_test = bundle["y_test"]

### Performance Monitoring (AUC Stability)

In [10]:
current_auc = roc_auc_score(y_test, y_pred_proba)
current_auc

np.float64(0.7765302827521237)

### Interpretation

- Model discrimination remains strong and stable

- No evidence of immediate performance decay

- Suitable for continued production use

### Business Meaning

The model continues to effectively rank customers by default risk, preserving decision quality for credit interventions.

In [11]:
train_scores = bundle["train_proba"]
test_scores = y_pred_proba

ks_stat, p_value = ks_2samp(train_scores, test_scores)

ks_stat, p_value

(np.float64(0.013600000000000001), np.float64(0.24693845928939206))

### Interpretation

- High p-value (> 0.05) → No statistically significant distribution shift

- Training and current scoring populations are aligned

### Business Meaning

Customer behavior patterns have not materially changed since model training.
No retraining or recalibration is required at this stage.

### PSI Calculation

In [12]:
def calculate_psi(expected, actual, bins=10):
    breakpoints = np.percentile(expected, np.linspace(0, 100, bins + 1))
    psi = 0
    for i in range(len(breakpoints) - 1):
        e_perc = np.mean((expected >= breakpoints[i]) & (expected < breakpoints[i+1]))
        a_perc = np.mean((actual >= breakpoints[i]) & (actual < breakpoints[i+1]))
        psi += (e_perc - a_perc) * np.log((e_perc + 1e-6) / (a_perc + 1e-6))
    return psi

psi_value = calculate_psi(train_scores, test_scores)
psi_value


np.float64(0.002418933158799369)

### PSI Thresholds
### PSI Range	Status
- < 0.10	Stable
- 0.10–0.25	Monitor
- > 0.25	Action Required
### Interpretation

- PSI = 0 → Extremely stable population

- Model predictions remain valid and reliable

### Business Meaning

The model is operating within expected statistical boundaries and meets governance standards for regulated financial systems.

The Credit Risk Early Warning Model is production-ready, statistically stable, and compliant with model risk management expectations.

Continuous monitoring thresholds are defined, and no immediate corrective action is required.